# Flu Serology & HAI — Held-Out Validation Block
**Not a BN training input** — serology/HAI are measured only at vaccine visits (structured
missingness), so this block is held out and used solely to *benchmark* the fitted network
(e.g. query `P(seroconversion | age_group=Older, CMV=Positive)` vs literature expectations).

Sources: `data/raw/sound-life_flu_serology_single.csv` (IgG), `data/raw/sound-life_flu_hai_single.csv` (HAI)
Output: `data/processed/flu_response_validation.csv` (standalone; joined to the fitted-network
subjects at benchmark time, never folded into `bn_ready_baseline.csv`)

**Design decisions:**
- **Anchor on `visitName`** (`Flu Year 1 Day 0/7/90`), NOT `vaccine.year` — vaccine.year is mixed
  within Flu Year 1 (2020-2021 + 2019-2020) and was applied inconsistently in the R draft.
- **One row per subject, keyed on `subject.subjectGuid` + Flu-Y1-Day-0 `sample.sampleKitGuid`**
  (= the spine key). D7/D90 kits + vaccine.year kept as provenance. D0 kits are all 92 spine kits.
- **Replicate collapse by mean** (matches `eda/flu_serology_hai.ipynb`, which pivots with default
  aggfunc='mean'). HAI `_single.csv` is per-sample-PER-PLATE: the same kit+specimen is measured on
  3 plates with wide disagreement -> `_replspread_flag` (IgG std/mean>0.20; HAI max-min>20pp).
- **IgG target:** per-antigen `fc = conc(D7)/conc(D0)`, `seroconvert = fc>=4`. **HAI target:**
  `peak = Day 7 % inhibition`. Day-0 pre-existing-immunity flags (>90th pct) for both (ceiling
  effect). IgG (`concMean`, 7 antigens) and HAI (`percentNormInhibition`, 9 antigens; BSA dropped)
  kept as separate blocks (panels differ — no forced antigen alignment).

## Stage 0 — Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))
from src.load import load_csv
from src.transforms import to_snake_case

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")

SUBJ = "subject.subjectGuid"
KIT = "sample.sampleKitGuid"
TPS = {"d0": "Flu Year 1 Day 0", "d7": "Flu Year 1 Day 7", "d90": "Flu Year 1 Day 90"}

IGG_ANTIGENS = ["A/Brisbane", "A/Hong Kong", "A/Michigan", "A/Victoria",
                "B/Colorado", "B/Phuket", "B/Washington"]
HAI_ANTIGENS = ["A/Brisbane", "A/Cambodia", "A/Guangdong", "A/Hong Kong", "A/Kansas",
                "A/Phuket", "A/Shanghai", "A/Wisconsin", "B/Washington"]   # BSA dropped

CV_GATE = 20.0          # % CV above which a measurement is flagged unreliable
SEROCONVERT_FC = 4.0    # >=4-fold rise = seroconversion
HIGH_PCTILE = 0.90      # Day-0 > cohort 90th pct = pre-existing-immunity flag
IGG_SPREAD = 0.20       # replicate std/mean threshold (IgG)
HAI_SPREAD = 20.0       # replicate max-min threshold, pp (HAI)

OUT = PROCESSED / "flu_response_validation.csv"

def masked_bool(cond, valid):
    """Nullable-boolean flag: True/False where `valid` is non-null, else <NA>."""
    return pd.Series(np.asarray(cond), index=valid.index).where(valid.notna()).astype("boolean")

## Stage 1 — Ingestion

In [2]:
sero = load_csv(RAW / "sound-life_flu_serology_single.csv")
hai = load_csv(RAW / "sound-life_flu_hai_single.csv")
print(f"serology: {sero.shape}  subjects={sero[SUBJ].nunique()}  antigens={sero['msd.antigenName'].nunique()}")
print(f"HAI:      {hai.shape}  subjects={hai[SUBJ].nunique()}  antigens={hai['msd.antigenName'].nunique()} (incl BSA)")

# Panel sanity: expected antigens present.
assert set(IGG_ANTIGENS) <= set(sero['msd.antigenName']), "missing IgG antigen"
assert set(HAI_ANTIGENS) <= set(hai['msd.antigenName']), "missing HAI antigen"
print("expected IgG(7) + HAI(9, BSA excluded) antigens present.")

serology: (3667, 27)  subjects=92  antigens=7
HAI:      (9140, 26)  subjects=96  antigens=10 (incl BSA)
expected IgG(7) + HAI(9, BSA excluded) antigens present.


## Stage 2 — Per-subject × antigen × timepoint Aggregation (mean + replicate stats)

Collapse plate replicates by mean; also retain std/min/max/count and max CV per group to derive
the reliability and replicate-disagreement flags.

In [3]:
def agg_stats(df, value_col, cv_col, antigens):
    """Per (subject, antigen, timepoint): mean value + std/min/max/count + max CV."""
    out = {}
    for tp, visit in TPS.items():
        sub = df[(df["sample.visitName"] == visit) & (df["msd.antigenName"].isin(antigens))]
        g = (sub.groupby([SUBJ, "msd.antigenName"])
             .agg(mean=(value_col, "mean"), std=(value_col, "std"),
                  vmin=(value_col, "min"), vmax=(value_col, "max"),
                  cvmax=(cv_col, "max"), n=(value_col, "count"))
             .reset_index())
        out[tp] = g
    return out

igg_stats = agg_stats(sero, "msd.concMean", "msd.concPercentCV", IGG_ANTIGENS)
hai_stats = agg_stats(hai, "msd.percentNormInhibition", "msd.signalNormPercentCV", HAI_ANTIGENS)

# replicate multiplicity actually collapsed (sanity)
for nm, st in [("IgG", igg_stats), ("HAI", hai_stats)]:
    d0 = st["d0"]
    print(f"{nm} D0 groups with >1 replicate collapsed: {int((d0['n'] > 1).sum())}/{len(d0)}")

IgG D0 groups with >1 replicate collapsed: 3/644
HAI D0 groups with >1 replicate collapsed: 72/828


## Stage 3 — Build the per-subject wide block

For each antigen: mean level per timepoint, per-timepoint CV flag (D0/D7) and replicate-spread
flag, then the derived targets.

In [4]:
def build_block(stats, antigens, prefix, spread_rule):
    subjects = sorted(stats["d0"][SUBJ].unique())
    W = pd.DataFrame({SUBJ: subjects})
    for ag in antigens:
        s = to_snake_case(ag)
        for tp in TPS:
            gg = stats[tp][stats[tp]["msd.antigenName"] == ag].set_index(SUBJ)
            mean = W[SUBJ].map(gg["mean"])
            W[f"val.{prefix}_{tp}_{s}"] = mean
            if tp in ("d0", "d7"):
                cvmax = W[SUBJ].map(gg["cvmax"])
                W[f"val.{prefix}_cv_flag_{tp}_{s}"] = masked_bool(cvmax > CV_GATE, cvmax)
            std = W[SUBJ].map(gg["std"]); vmin = W[SUBJ].map(gg["vmin"])
            vmax = W[SUBJ].map(gg["vmax"]); n = W[SUBJ].map(gg["n"])
            disagree = spread_rule(mean, std, vmin, vmax) & (n > 1)
            W[f"val.{prefix}_{tp}_replspread_flag_{s}"] = masked_bool(disagree, mean)
    return W

igg_block = build_block(igg_stats, IGG_ANTIGENS, "igg",
                        lambda m, sd, lo, hi: (sd / m) > IGG_SPREAD)
hai_block = build_block(hai_stats, HAI_ANTIGENS, "hai",
                        lambda m, sd, lo, hi: (hi - lo) > HAI_SPREAD)
print(f"IgG block: {igg_block.shape}   HAI block: {hai_block.shape}")

IgG block: (92, 57)   HAI block: (92, 73)


## Stage 4 — Derived Validation Targets

IgG: fold change (D7/D0), log2 FC, seroconversion (>=4-fold), Day-0 high-titer flag.
HAI: peak = Day 7 inhibition, Day-0 high flag. (No HAI fold change — inhibition is bounded/signed.)

In [5]:
for ag in IGG_ANTIGENS:
    s = to_snake_case(ag)
    d0 = igg_block[f"val.igg_d0_{s}"]; d7 = igg_block[f"val.igg_d7_{s}"]
    fc = pd.Series(np.where(d0.isna() | (d0 == 0), np.nan, d7 / d0), index=igg_block.index)
    igg_block[f"val.igg_fc_{s}"] = fc
    igg_block[f"val.igg_log2fc_{s}"] = np.where(fc > 0, np.log2(fc.where(fc > 0, 1)), np.nan)
    igg_block[f"val.igg_seroconvert_{s}"] = masked_bool(fc >= SEROCONVERT_FC, fc)
    thr = d0.quantile(HIGH_PCTILE)
    igg_block[f"val.igg_d0_hightiter_flag_{s}"] = masked_bool(d0 > thr, d0)

for ag in HAI_ANTIGENS:
    s = to_snake_case(ag)
    d0 = hai_block[f"val.hai_d0_{s}"]; d7 = hai_block[f"val.hai_d7_{s}"]
    hai_block[f"val.hai_peak_{s}"] = d7
    thr = d0.quantile(HIGH_PCTILE)
    hai_block[f"val.hai_d0_high_flag_{s}"] = masked_bool(d0 > thr, d0)

sc_cols = [c for c in igg_block.columns if "_seroconvert_" in c]
rate = pd.concat([igg_block[c].astype("float") for c in sc_cols]).mean()
print(f"overall seroconversion rate (mean over antigens x subjects): {rate:.3f}")

overall seroconversion rate (mean over antigens x subjects): 0.084


## Stage 5 — Keys, Provenance, Assembly

Key = subject + Flu-Y1-Day-0 kit (= spine key). D7/D90 kits + vaccine.year as provenance. Assert
IgG and HAI agree on the Day-0 kit per subject (same physical sample).

In [6]:
def kit_map(df, visit):
    x = df[df["sample.visitName"] == visit][[SUBJ, KIT]].drop_duplicates()
    assert not x[SUBJ].duplicated().any(), f"{visit}: >1 kit per subject"
    return x.set_index(SUBJ)[KIT]

# Day-0 kit from serology is the canonical key; assert HAI agrees where both exist.
d0_sero = kit_map(sero, "Flu Year 1 Day 0")
d0_hai = kit_map(hai, "Flu Year 1 Day 0")
common = d0_sero.index.intersection(d0_hai.index)
assert (d0_sero.loc[common] == d0_hai.loc[common]).all(), "IgG/HAI disagree on Day-0 kit"

subjects = sorted(set(igg_block[SUBJ]) | set(hai_block[SUBJ]))
base = pd.DataFrame({SUBJ: subjects})
base[KIT] = base[SUBJ].map(d0_sero)
base["sample.d7_sampleKitGuid"] = base[SUBJ].map(kit_map(sero, "Flu Year 1 Day 7"))
base["sample.d90_sampleKitGuid"] = base[SUBJ].map(kit_map(sero, "Flu Year 1 Day 90"))
base["vaccine.year"] = base[SUBJ].map(
    sero[sero["sample.visitName"] == "Flu Year 1 Day 0"].groupby(SUBJ)["vaccine.year"].first())

val = base.merge(igg_block, on=SUBJ, how="left").merge(hai_block, on=SUBJ, how="left")
print(f"assembled: {val.shape}  subjects={val[SUBJ].nunique()}")
assert base[KIT].notna().all(), "subject without a Day-0 kit"
assert not val[SUBJ].duplicated().any(), "duplicate subject"
print(f"Day-0 key kits unique: {val[KIT].is_unique}")

assembled: (92, 179)  subjects=92
Day-0 key kits unique: True


## Stage 6 — Order, Missingness, Export

In [7]:
def is_flag(c): return ("_flag_" in c) or c.endswith("_flag")

keys = [SUBJ, KIT]
prov = ["sample.d7_sampleKitGuid", "sample.d90_sampleKitGuid", "vaccine.year"]
igg_feat = [c for c in val.columns if c.startswith("val.igg_") and not is_flag(c)]
hai_feat = [c for c in val.columns if c.startswith("val.hai_") and not is_flag(c)]
flag_cols = [c for c in val.columns if is_flag(c)]
order = keys + prov + igg_feat + hai_feat + flag_cols
assert set(order) == set(val.columns), f"ordering drift: {set(val.columns) ^ set(order)}"
val = val[order]

# Drop only 100%-empty columns (auditable); no imputation.
empty = [c for c in val.columns if val[c].notna().sum() == 0]
print(f"100%-empty columns dropped: {empty if empty else 'none'}")
val = val.drop(columns=empty)

# Missingness on validation targets (report only).
target_cols = [c for c in val.columns if c.startswith("val.") and not is_flag(c)]
miss = val[target_cols].isna().mean()
print(f"\nvalidation target columns: {len(target_cols)}  mean missing: {miss.mean():.1%}")
print("most-missing targets:")
print(miss.sort_values(ascending=False).head(5).to_string())

# Sanity vs spine.
spine_kits = set(load_csv(PROCESSED / 'clinical_baseline_wide.csv')[KIT])
in_spine = val[KIT].isin(spine_kits).sum()
print(f"\nDay-0 kits in spine: {in_spine}/{len(val)}")

# Contract: keys first, unique, non-null.
assert list(val.columns[:2]) == keys and val[KIT].is_unique and val[keys].notna().all().all()
val.to_csv(OUT, index=False)
print(f"written: {OUT}  ({val.shape[0]} x {val.shape[1]})")

100%-empty columns dropped: none

validation target columns: 78  mean missing: 0.7%
most-missing targets:
val.igg_d90_a_hong_kong    0.032609
val.igg_d90_a_brisbane     0.032609
val.igg_d90_a_victoria     0.032609
val.igg_d90_a_michigan     0.032609
val.hai_d90_a_wisconsin    0.032609



Day-0 kits in spine: 92/92
written: ..\data\processed\flu_response_validation.csv  (92 x 179)


In [8]:
# --- hand-off summary ---
n_flags = sum(1 for c in val.columns if is_flag(c))
print("=== flu_response_validation.csv HAND-OFF (held-out benchmark) ===")
print(f"rows (subjects):        {val.shape[0]}")
print(f"cols (total):           {val.shape[1]}")
print(f"  IgG features:         {len([c for c in val.columns if c.startswith('val.igg_') and not is_flag(c)])}")
print(f"  HAI features:         {len([c for c in val.columns if c.startswith('val.hai_') and not is_flag(c)])}")
print(f"  flag columns (last):  {n_flags}")
print(f"join key:               {SUBJ} + {KIT} (Flu-Y1-Day-0)")
print(f"output:                 {OUT}")
print("\nHeld out from BN training; benchmark by querying the fitted network and comparing to")
print("val.igg_seroconvert_* / val.igg_fc_* / val.hai_peak_* under known age/CMV expectations.")
val.iloc[:5, :7]

=== flu_response_validation.csv HAND-OFF (held-out benchmark) ===
rows (subjects):        92
cols (total):           179
  IgG features:         42
  HAI features:         36
  flag columns (last):  96
join key:               subject.subjectGuid + sample.sampleKitGuid (Flu-Y1-Day-0)
output:                 ..\data\processed\flu_response_validation.csv

Held out from BN training; benchmark by querying the fitted network and comparing to
val.igg_seroconvert_* / val.igg_fc_* / val.hai_peak_* under known age/CMV expectations.


,subject.subjectGuid,sample.sampleKitGuid,sample.d7_sampleKitGuid,sample.d90_sampleKitGuid,vaccine.year,val.igg_d0_a_brisbane,val.igg_d7_a_brisbane
0,BR1001,KT00001,KT00008,KT00037,2019-2020,2327.356533,2497.469720
1,BR1002,KT00002,KT00009,KT00034,2019-2020,1911.125338,2262.986249
2,BR1003,KT00003,KT00007,KT00035,2019-2020,575.481973,1835.167711
3,BR1004,KT00004,KT00011,KT00265,2019-2020,632.716966,2797.063715
4,BR1005,KT00006,KT00013,KT00040,2019-2020,2403.628517,2798.520432
